In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

sys.path.insert(0, "..")  # repo root, so `core` is importable from notebooks/

from core.offline import collect_offline_trajectories
from core.radius import estimate_radius, fit_predictor
from core.mpc import make_controller, simulate, closed_loop_metrics

## Define the system matrices and parameters

In [ ]:
A = np.array([[0.9, 0.1], [0.05, 0.9]])
B = np.array([[0.0], [1.0]])
n, m, T = 2, 1, 5
spec = dict(state_weight=[1, 0], ref=[2, 0], x_min=[-np.inf, 0], x_max=[1, np.inf])

## Collect offline trajectories and estimate the radius of uncertainty

In [ ]:
rng = np.random.default_rng(0)
Z, Y = collect_offline_trajectories(A, B, 10, T, rng=rng)
L_hat, eps1, eps2 = estimate_radius(Z, Y, n, m)
W = rng.normal(0, 0.03, (30, n))

def run(L_hat, Z, Y, eps1, eps2, W):
    """Closed-loop (cost, violations) from the paper's x0; (nan, nan) if the run fails."""
    try:
        controller = make_controller(L_hat, Z, Y, n, m, eps1, eps2, **spec)
        X, _ = simulate(A, B, controller, [0.9, 0.9], W)
        return closed_loop_metrics(X, **spec)
    except RuntimeError:
        return np.nan, np.nan

In [ ]:
cost, violations = run(L_hat, Z, Y, eps1, eps2, W)

In [ ]:
cost

In [ ]:
violations

## Fig. 1: effect of the radius

The paper fixes one dataset of size N=10 and one noise realization, then runs every combination of eps1 and eps2 in {1e-7, ..., 1}. The paper's dataset isn't published, so cell values won't match exactly; the comparison is about the overall pattern.

This plot uses seed 9, the closest of seeds 0-19 to the paper by mean cost error (0.77). That makes it a selected dataset, not a typical one: 5 of those 20 (seeds 0, 2, 6, 14, 16) misbehave at the smallest radius, with failed runs, cost above 100, or 29 violations.

In [ ]:
EPS_GRID = 10.0 ** np.arange(-7, 1)

# Separate RNG, so the Fig. 2 sweep below is unaffected.
fig1_rng = np.random.default_rng(9)
Z, Y = collect_offline_trajectories(A, B, 10, T, rng=fig1_rng)
W = fig1_rng.normal(0, 0.03, (30, n))

L_fit = fit_predictor(Z, Y, n, m)  # eq. 13; Fig. 1 uses least squares, not Algorithm 1
grid = np.array([[run(L_fit, Z, Y, e1, e2, W) for e1 in EPS_GRID] for e2 in EPS_GRID])
cost_grid, violation_grid = grid[..., 0], grid[..., 1]

BLUES = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#6da7ec", "#256abf", "#104281"])
labels = [f"$10^{{{e}}}$" for e in range(-7, 1)]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), facecolor="#fcfcfb")
for ax, values, title, fmt in [
    (axes[0], cost_grid, "Closed-loop cost", "{:.1f}"),
    (axes[1], violation_grid, "Closed-loop violations", "{:.0f}"),
]:
    image = ax.imshow(values, cmap=BLUES)
    midpoint = np.nanmean([np.nanmin(values), np.nanmax(values)])
    for (i, j), value in np.ndenumerate(values):
        text = "fail" if np.isnan(value) else fmt.format(value)
        ax.text(j, i, text, ha="center", va="center", fontsize=8,
                color="#ffffff" if value > midpoint else "#0b0b0b")
    ax.set_xticks(range(8), labels)
    ax.set_yticks(range(8), labels)
    ax.set_xlabel("$\\varepsilon_1$")
    ax.set_ylabel("$\\varepsilon_2$")
    ax.set_title(title)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

## Fig. 2: SAAMPC vs. DRMPC by dataset size

For N = 10, 20, 30 and 40, we repeat 50 times with a new dataset and noise realization. DRMPC uses the Algorithm 1 radius and SAAMPC sets the radius to 0, with the same predictor, data and noise. A run counts as failed if the solver gives up, which usually means the closed loop diverged.

In [ ]:
results = {}
for num_trajectories in [10, 20, 30, 40]:
    runs = {"SAAMPC": [], "DRMPC": []}

    for trial in range(50):
        Z, Y = collect_offline_trajectories(A, B, num_trajectories, T, rng=rng)
        L_hat, eps1, eps2 = estimate_radius(Z, Y, n, m)
        W = rng.normal(0, 0.03, (30, n))
        runs["SAAMPC"].append(run(L_hat, Z, Y, 0.0, 0.0, W))
        runs["DRMPC"].append(run(L_hat, Z, Y, eps1, eps2, W))

    results[num_trajectories] = {name: np.array(r) for name, r in runs.items()}
    for name, r in results[num_trajectories].items():
        costs, violations = r.T
        print(
            f"N={num_trajectories} {name:<6}: failed {np.isnan(costs).sum()}/50 | "
            f"cost mean {np.nanmean(costs):.2f}, median {np.nanmedian(costs):.2f}, "
            f"range {np.nanmin(costs):.1f}-{np.nanmax(costs):.1f} | "
            f"violations mean {np.nanmean(violations):.2f}, median {np.nanmedian(violations):.1f}, "
            f"zero in {(violations == 0).sum()}/50"
        )

In [ ]:
COLORS = {"SAAMPC": "#2a78d6", "DRMPC": "#eb6834"}
fig, axes = plt.subplots(4, 2, figsize=(10, 8), sharey=True, facecolor="#fcfcfb")
panels = [(0, np.arange(15, 41), "CL cost"), (1, np.arange(0, 31), "Number of CL violations")]

for row, (num_trajectories, by_name) in zip(axes, results.items()):
    for ax, (column, bins, xlabel) in zip(row, panels):
        notes = []
        for name, r in by_name.items():
            values = r[:, column]
            ax.hist(values, bins=bins, color=COLORS[name], alpha=0.65, label=f"{name} N={num_trajectories}")
            failed = np.isnan(values).sum()
            # The paper's axis limits can hide runs, so say how many fall outside them.
            off_axis = ((values < bins[0]) | (values > bins[-1])).sum()
            if failed or off_axis:
                notes.append(f"{name}: {off_axis} off axis, {failed} failed")
        ax.set_xlim(bins[0], bins[-1])
        ax.set_ylim(0, 50)
        ax.set_ylabel("Count")
        ax.legend(loc="upper left" if column == 0 else "upper right", frameon=False, fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["left", "bottom"]].set_color("#c3c2b7")
        if notes:
            ax.text(0.5, 0.92, "\n".join(notes), transform=ax.transAxes, ha="center", va="top",
                    fontsize=8, color="#52514e")

axes[0, 0].set_title("Closed-loop cost")
axes[0, 1].set_title("Closed-loop violations")
axes[-1, 0].set_xlabel(panels[0][2])
axes[-1, 1].set_xlabel(panels[1][2])
fig.tight_layout()
plt.show()